# Who Wants to Be a PoliMillionaire? — NLP Group Assignment 2025/26

| | |
|---|---|
| **Group members** | _Name 1_ — `email1@mail.polimi.it`&nbsp;&nbsp;·&nbsp;&nbsp;_Name 2_ — `email2@mail.polimi.it`&nbsp;&nbsp;·&nbsp;&nbsp;_Name 3_ — `email3@mail.polimi.it` |
| **Video link** | _paste URL here_ |
| **Model family** | Qwen 2.5 (7B-Instruct · 14B-Instruct, 4-bit NF4) |
| **Best strategy** | Tiered hybrid — baseline → RAG → ensemble · Maths → agent |

---

**Coding-assistant disclosure.**
Scaffold architecture, configuration patterns, retrieval/RAG glue, runner, and evaluation harness structure were drafted with assistance from an AI coding assistant (Claude, Anthropic). Architectural decisions, model selection, prompt engineering, ablation experimental design, the analytical narrative, and all reported results were produced by the human group members. No portion of the assignment was delegated wholesale to any LLM.

## 0. System Architecture

We build a **tiered hybrid chatbot** that routes each question to the cheapest capable strategy.

```
Game API  ──►  TieredStrategy (router)
               ├─ L 1– 5  ──►  BaselineLLM      fast logit-scoring        ~1–2 s
               ├─ L 6–10  ──►  RAGStrategy       Wikipedia + DDG + FAISS   ~3–5 s
               ├─ L11–15  ──►  EnsembleStrategy  2× LLM prob-fusion        ~5–8 s
               └─ Maths   ──►  AgentStrategy     ReAct + MathsTool sandbox ~3–6 s
                    │
                    │  confidence-aware escalation
                    │  if margin < θ → try next tier
                    ▼
               EvalHarness (offline replay from JSONL logs)
               ├─ accuracy + per-category breakdown
               ├─ ECE (expected calibration error)
               └─ latency p50 / p95
```

**Design principles:**
- All Python lives in `polimibot/` package — notebook is narration and assembly only
- Logit-scoring preferred over free generation — deterministic, one forward pass, no output parsing
- Two-phase RAG — build index once offline, retrieve at inference time
- JSONL run logs enable offline evaluation without replaying against the live API
- `@dataclass(frozen=True)` for all config and DTOs — mutation bugs caught at construction


In [ ]:
import os

token = os.getenv("GITHUB_TOKEN")

repo_url = 'github.com/m-ebrahimzadeh/PoliMillionaire'

# Clone the private repository using the token for authentication
!git clone https://{token}@{repo_url}

In [4]:
# === Environment detection and repository path setup. Know where we are, we must. ===
import os
import sys
from pathlib import Path

# Detect Colab — present in sys.modules only after a Colab-specific import has run.
# Safer fallback: check for the /content directory that Colab always creates.
_IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if _IN_COLAB:
    from google.colab import drive
    # Mount Drive only once — idempotent this block is, but the mount call is not.
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    # ── Adjust this path to match where your group stored the repo on Drive. ──
    _REPO = Path('/content/drive/MyDrive/Colab Notebooks/Polimillionaire')
    if not _REPO.exists():
        # Cloned directly into /content — also valid, this path is.
        _REPO = Path('/content/PoliMillionaire')
    if _REPO.exists():
        os.chdir(_REPO)
        if str(_REPO) not in sys.path:
            sys.path.insert(0, str(_REPO))
        print(f'Repo root set: {_REPO}')
    else:
        print('WARNING: repo not found. Set _REPO manually, you must.')
else:
    # Local dev: the notebook lives at the repo root — correct, this assumption is.
    _REPO = Path('.').resolve()
    if str(_REPO) not in sys.path:
        sys.path.insert(0, str(_REPO))

print(f'{"Colab" if _IN_COLAB else "Local"} mode  |  cwd: {Path.cwd()}  |  Python {sys.version.split()[0]}')

Repo root set: /content/drive/MyDrive/Colab Notebooks/Polimillionaire
Colab mode  |  cwd: /content/drive/.shortcut-targets-by-id/1KcAneinPOufzpKEpu0JwBKwDlrzFJ1Vr/Polimillionaire  |  Python 3.12.13


In [8]:
# === Install dependencies. Once per runtime, run this cell you should. ===
# Heavy packages: transformers, bitsandbytes (quantization), faiss, sentence-transformers.
# Subsequent cells import from cache — fast, the second run is.
import importlib.util
import importlib.metadata

def is_package_installed(package_list):
    # Return True only if ALL packages in the list are found
    for pack in package_list:
        # Some packages have different distribution names than import names
        dist_name = pack.replace('_', '-')
        try:
            importlib.metadata.version(dist_name)
        except importlib.metadata.PackageNotFoundError:
            return False
    return True

# The required packages we want to track
packages = [
    "transformers", "accelerate", "bitsandbytes",
    "sentence_transformers", "faiss-cpu", "duckduckgo-search",
    "trafilatura", "sympy", "matplotlib", "pandas", "tqdm"
]

# Map import names to distribution names for accurate version checking
package_map = {
    "transformers": "transformers",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
    "sentence_transformers": "sentence-transformers",
    "faiss": "faiss-cpu",
    "duckduckgo_search": "duckduckgo-search",
    "trafilatura": "trafilatura",
    "sympy": "sympy",
    "matplotlib": "matplotlib",
    "pandas": "pandas",
    "tqdm": "tqdm"
}

if not is_package_installed(list(package_map.values())):
    print("Dependencies not found, installing...")
    %pip install -q \
        "transformers==4.46.*" "accelerate==1.1.*" "bitsandbytes>=0.43" \
        "sentence-transformers==3.2.*" "faiss-cpu==1.9.*" \
        "duckduckgo-search==6.3.*" "trafilatura==1.12.*" \
        "sympy==1.13.*" matplotlib pandas tqdm
else:
    print("Dependencies already installed. Skipping installation.")

print("\n--- Package Versions ---")
for display_name, dist_name in package_map.items():
    try:
        version = importlib.metadata.version(dist_name)
        print(f"{display_name:<20}: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{display_name:<20}: Not found")

Dependencies already installed. Skipping installation.

--- Package Versions ---
transformers        : 4.46.3
accelerate          : 1.1.1
bitsandbytes        : 0.49.2
sentence_transformers: 3.2.1
faiss               : 1.9.0.post1
duckduckgo_search   : 6.3.7
trafilatura         : 1.12.2
sympy               : 1.13.3
matplotlib          : 3.10.0
pandas              : 2.2.2
tqdm                : 4.67.3


In [5]:

# === Redirect HuggingFace cache to Google Drive. Re-download every session, you will not. ===
import os

HF_CACHE_DIR = os.path.join(_REPO, 'hf_cache')
os.makedirs(HF_CACHE_DIR, exist_ok=True)

# These two lines are the magic — must be set before any model loads
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['TRANSFORMERS_CACHE'] = HF_CACHE_DIR
os.environ['HF_DATASETS_CACHE'] = os.path.join(HF_CACHE_DIR, 'datasets')

print(f"HuggingFace cache redirected to Drive: {HF_CACHE_DIR}")
print("First session: models will download. After that: instant load from Drive.")

HuggingFace cache redirected to Drive: /content/drive/MyDrive/Colab Notebooks/Polimillionaire/hf_cache
First session: models will download. After that: instant load from Drive.


In [9]:
# === Install the polimibot package in editable mode. The heart of our system, this is. ===
import subprocess
_result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', str(_REPO)],
    capture_output=True, text=True
)
if _result.returncode != 0:
    # Graceful fallback: sys.path already set — imports may still work.
    print('Editable install failed (non-fatal if imports work):')
    print(_result.stderr[:400])
else:
    print('polimibot installed ✓')

polimibot installed ✓


In [10]:
# === Core imports and user configuration. Assemble the tools, we now do. ===
import json
import os
import time
import warnings
from pathlib import Path
from typing import List, Optional

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')  # Suppress noisy transformer warnings — distract us, they do.

# ── Our package ──────────────────────────────────────────────────────────────
from polimibot.config import PATHS, RUNTIME, Category
from polimibot.eval.calibration import calibration_from_gold_set, plot_calibration
from polimibot.eval.evaluator import evaluate_strategy, EvalReport
from polimibot.eval.gold_set import harvest_gold_set, load_gold_set
from polimibot.eval.report_io import save_report
from polimibot.models.llm import load_llm
from polimibot.prompts.templates import PromptStyle
from polimibot.strategies.agent_strategy import AgentStrategy
from polimibot.strategies.ensemble_strategy import EnsembleStrategy
from polimibot.strategies.llm_baseline import BaselineLLMStrategy
from polimibot.strategies.rag_strategy import RAGStrategy
from polimibot.strategies.tiered_strategy import TierBreakpoints, TieredStrategy
from polimibot.tools.maths_tool import MathsTool

# ── User-configurable constants — fill in these, your group must. ─────────────
USERNAME  = 'fou'   # ← change this
PASSWORD  = '3501744306'   # ← change this
BASE_URL  = 'http://131.175.15.22:51111'

# ── Ensure all output directories exist — before we write, create them we must. ─
PATHS.ensure()

# Quick sanity check: loudly, path bugs must fail — silently, acceptable they are not.
print(f'data root : {PATHS.data_dir}')
print(f'eval dir  : {PATHS.eval_dir}')
print(f'runs dir  : {PATHS.runs_dir}')
print(f'cache dir : {PATHS.cache_dir}')
print('All imports successful ✓')

data root : /content/drive/.shortcut-targets-by-id/1KcAneinPOufzpKEpu0JwBKwDlrzFJ1Vr/Polimillionaire/data
eval dir  : /content/drive/.shortcut-targets-by-id/1KcAneinPOufzpKEpu0JwBKwDlrzFJ1Vr/Polimillionaire/data/eval
runs dir  : /content/drive/.shortcut-targets-by-id/1KcAneinPOufzpKEpu0JwBKwDlrzFJ1Vr/Polimillionaire/data/runs
cache dir : /content/drive/.shortcut-targets-by-id/1KcAneinPOufzpKEpu0JwBKwDlrzFJ1Vr/Polimillionaire/data/cache
All imports successful ✓


## 1. Game Connection

We connect to the PoliMillionaire API and enumerate available competitions.

**Architecture note:** The `GameAdapter` class inside `polimibot/game/adapter.py` is the **single file** in our codebase that directly touches the `millionaire_client` library. Every other module receives a typed `GameAdapter` — never the raw client.

This means if the API changes, we edit exactly **one file**. This pattern is called a *façade* or *anti-corruption layer*.

In [11]:
# === Connect to the game server. The gateway to the competition, this is. ===
from polimibot.game.adapter import connect
client = connect(BASE_URL, USERNAME, PASSWORD)


# Discover all competitions — know our battlefield, we must.
competitions  = client.competitions.list_all()
CATEGORIES    = {c.id: c for c in competitions}

print(f'Connected as: {USERNAME}')
print(f'\nAvailable competitions ({len(CATEGORIES)}):')
for cid, comp in sorted(CATEGORIES.items()):
     print(f'  [{cid}] {comp.name:<25s}  levels 1–{comp.max_levels}')


Connected as: fou

Available competitions (4):
  [0] Entertainment              levels 1–15
  [1] Ancient History and Politics  levels 1–15
  [2] Science and Nature         levels 1–15
  [3] Maths                      levels 1–15


### 1.1 Game Session Helper

`play_session` wraps the game runner: plays N games per competition, saves every question
to a JSONL log, respects the API courtesy delay, and returns typed `GameSummary` objects.
Defined once here; all subsequent sections call it with different strategies.

In [12]:
# === Game session helper. Play games and record everything, this function does. ===
from dataclasses import dataclass

@dataclass
class GameSummary:
    # The record of one completed game — stored here, the truth is.
    competition_id   : int
    competition_name : str
    session_id       : int
    final_level      : int
    earned_amount    : float
    accuracy         : float
    elapsed_seconds  : float
    strategy_name    : str
    n_questions      : int


def play_session(
    client,
    competition_ids : List[int],
    strategy,
    games_per_competition : int = 1,
    run_name              : str  = 'session',
    verbose               : bool = True,
) -> List[GameSummary]:
    # Orchestrate multiple games across competitions — the outer loop, this is.
    from polimibot.runner import GameRunner

    strategy.warm_up()   # Load weights into VRAM once — cold start, avoid we must.

    summaries = []
    for cid in competition_ids:
        comp = CATEGORIES[cid]
        for g in range(games_per_competition):
            if verbose:
                print(f'  [{comp.name}]  game {g+1}/{games_per_competition} …', end='', flush=True)

            runner  = GameRunner(client=client, competition_id=cid,
                                 strategy=strategy,
                                 run_name=f'{run_name}_c{cid}_g{g}')
            summary = runner.run()

            summaries.append(GameSummary(
                competition_id   = cid,
                competition_name = comp.name,
                session_id       = summary.session_id,
                final_level      = summary.final_level,
                earned_amount    = summary.earned_amount,
                accuracy         = summary.accuracy,
                elapsed_seconds  = summary.elapsed_seconds,
                strategy_name    = strategy.name,
                n_questions      = summary.n_questions,
            ))

            if verbose:
                print(f'  L{summary.final_level}  ${summary.earned_amount:,.0f}'
                      f'  acc={summary.accuracy:.0%}  {summary.elapsed_seconds:.1f}s')

            # Courtesy delay — the server, respect we must.
            time.sleep(RUNTIME.api_min_delay_seconds)

    strategy.shutdown()
    return summaries


print('play_session helper defined ✓')

play_session helper defined ✓


## 2. LLM Loading

### 💡 What is 4-bit quantization?

A 7-billion-parameter model stores each weight as a 32-bit float by default. On a Colab T4 GPU
(16 GB VRAM), that would require **~28 GB** — impossible.

**4-bit NF4 quantization** (Normal Float 4) compresses each weight to 4 bits by mapping it to
one of 16 values chosen specifically for the bell-curve distribution of neural network weights.

> *Analogy:* instead of storing a colour as `#3A7BCC` (24-bit), we store it as one of 16
> named crayon colours. The picture looks almost identical; the file is 6× smaller.

**Result:** 7B model fits in ~**5 GB** VRAM; 14B fits in ~**10 GB**. Both run on T4.

We load **two models**:
| Role | Model | VRAM | Used by |
|---|---|---|---|
| `fast_llm` | Qwen2.5-7B-Instruct | ~5 GB | baseline, RAG, agent, tiered-easy/medium |
| `smart_llm` | Qwen2.5-14B-Instruct | ~10 GB | ensemble hard tier (falls back to `fast_llm` if OOM) |


In [8]:
# === Load the fast (7B) LLM. The primary brain of our system, this is. ===
# 4-bit NF4 quantization: fits on T4, degrades accuracy only marginally it does.
fast_llm = load_llm(
    model_name    = 'Qwen/Qwen2.5-7B-Instruct',
    quantize_4bit = True,
    device        = 'auto',        # Let accelerate map layers to GPU/CPU — wise, this is.
    max_new_tokens= 16,            # Cap free-generation paths — logit-scoring ignores this.
)



Loading Qwen/Qwen2.5-7B-Instruct  (4bit=True)…


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loaded in 381.3s


In [13]:
print(f'Loaded : {fast_llm.model_name}')
print(f'Device : {fast_llm.device}')
print(f'VRAM   : {fast_llm.vram_gb:.1f} GB (estimated)')

NameError: name 'fast_llm' is not defined

In [14]:
# === Load the smart (14B) LLM for the ensemble tier. Optional, this model is. ===
# On T4 after 7B load (~5 GB used): ~10 GB free — enough for 14B-4bit (~10 GB).
# Conservative threshold used — OOM mid-load, worse than not loading, it is.

smart_llm = None

try:
    import torch
    if torch.cuda.is_available():
        _free_gb = (torch.cuda.get_device_properties(0).total_memory
                    - torch.cuda.memory_allocated(0)) / 1e9
        print(f'VRAM free: {_free_gb:.1f} GB')

        if _free_gb > 9.0:
            smart_llm = load_llm(
                model_name    = 'Qwen/Qwen2.5-14B-Instruct',
                quantize_4bit = True,
                device        = 'auto',
            )
            print(f'Smart LLM loaded: {smart_llm.model_name}  ({smart_llm.vram_gb:.1f} GB)')
        else:
            print(f'Insufficient VRAM ({_free_gb:.1f} GB) — reusing 7B for ensemble.')
            smart_llm = fast_llm
    else:
        print('No CUDA — CPU inference only. Slow, this will be.')
        smart_llm = fast_llm
except Exception as _e:
    print(f'14B load failed ({_e}) — ensemble will reuse fast_llm.')
    smart_llm = fast_llm

#print(f'Ensemble will use: fast={fast_llm.model_name} / smart={smart_llm.model_name}')

VRAM free: 15.6 GB
Loading Qwen/Qwen2.5-14B-Instruct  (4bit=True)…


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Loaded in 637.0s
Smart LLM loaded: Qwen/Qwen2.5-14B-Instruct  (9.1 GB)


NameError: name 'fast_llm' is not defined

## 3. Baseline Strategy

### 💡 What is logit scoring?

Standard multiple-choice solving would be: ask the model to write the answer, then parse its
response. This is fragile — the model might write "I think the answer is B, but actually
probably C...".

**Logit scoring** sidesteps generation entirely:

1. Build a prompt that ends with `"The answer is: "`
2. Run **one forward pass** (no token is generated)
3. Read the raw log-probabilities ("logits") for the four letter tokens: `A`, `B`, `C`, `D`
4. Apply softmax → proper probability distribution
5. Return the letter with the highest probability

```
P(A) = 0.05
P(B) = 0.82  ← winner
P(C) = 0.10
P(D) = 0.03
```

**Why this is better than free generation:**
| | Logit scoring | Free generation |
|---|---|---|
| Speed | One forward pass | N forward passes |
| Determinism | ✅ always same answer | ❌ temperature adds noise |
| Parsing | ✅ never fails | ❌ "I think it's..." |
| Confidence | ✅ true probability | ⚠️ requires extra parsing |

Our entire system — baseline, RAG, ensemble — uses logit scoring. Free generation is only used
as a fallback inside the `AgentStrategy` ReAct loop.


In [15]:
# === Build baseline strategy and run a smoke test. Trust but verify, we must. ===
from polimibot.strategies.base import StrategyInput
from polimibot.config import Category

baseline_zs = BaselineLLMStrategy(
    llm   = smart_llm,
    style = PromptStyle.ZERO_SHOT,
)

# Smoke test: a question with an obvious correct answer — fooled, the model should not be.
_test_q = StrategyInput(
    question = 'What is the capital of France?',
    options  = ('Berlin', 'Paris', 'Rome', 'Madrid'),
    level    = 1,
    category = Category.HISTORY,
)

_ans = baseline_zs.answer(_test_q)
_chosen = _test_q.options[_ans.chosen_index]
_correct = _chosen == 'Paris'
print(f'Question  : {_test_q.question}')
print(f'Chosen    : {_chosen}  (confidence {_ans.confidence:.3f})')
print(f'Correct   : {"✓" if _correct else "✗"}')
print(f'Strategy  : {baseline_zs.name}')


Question  : What is the capital of France?
Chosen    : Berlin  (confidence 0.561)
Correct   : ✗
Strategy  : baseline[qwen2.5-14b-instruct|zero_shot]


## 4. Building the Offline Evaluation Gold Set

### Why not measure accuracy from live games?

Live games are a **poor scientific instrument** for comparing strategies because:
- 🎲 **Different questions each run** — harder strategies get harder questions by design
- 🌐 **API latency adds noise** to timing measurements
- 💸 **A wrong answer ends the game** — we can't compare strategies on identical inputs

### The gold set approach

1. **Harvest**: play a few games; log every `(question, correct_answer, confidence)` to JSONL
2. **Build**: deduplicate by `(competition_id, question_text)` → `data/eval/gold_set.jsonl`
3. **Replay**: every strategy is evaluated on this fixed set — no live API calls, fair comparison

This is the standard **offline evaluation** methodology in NLP research. The gold set is built
**once** and never rebuilt from the same runs.

### Key metric: ECE (Expected Calibration Error)

Beyond accuracy, we measure **calibration** — does the model's confidence match its empirical
accuracy? A model that says "90% confident" should be right 90% of the time.

$$\text{ECE} = \sum_{b=1}^{B} \frac{|B_b|}{n} \left| \text{acc}(B_b) - \text{conf}(B_b) \right|$$

ECE = 0 is perfect. ECE = 0.2 means the model's stated confidence is on average 20% off from
its actual accuracy. A well-calibrated model makes better escalation decisions in the tiered strategy.


In [19]:
from polimibot.runner import play_game
result = play_game(client, competition_id=list(CATEGORIES.keys())[3], strategy=baseline_zs, verbose=True)
print(f'Final level: {result.final_level}, n_questions: {result.n_questions}, n_correct: {result.n_correct}')



=== Maths | strategy=baseline[qwen2.5-14b-instruct|zero_shot] ===

--- L1 (budget 25.0s) ---
Let S, T, and U be nonempty sets, and let f: S -> T and g: T -> U be functions such that the function g ∘ f : S -> U is one-to-one (injective). Which of the following must be true?
  A. f is one-to-one.
  B. g is onto.
  C. f is onto.
  D. g is one-to-one.
  → A  ✓  (3.13s)

--- L2 (budget 25.0s) ---
Statement 1 | If a and b are elements of finite order in an Abelian group, then |ab| is the lcm (|a|,|b|). Statement 2 | If g is a group element and g^n = e, then |g| = n.
  A. True, True
  B. False, True
  C. False, False
  D. True, False
  → A  ✗  (3.15s)
Final level: 2, n_questions: 2, n_correct: 1


In [13]:
# === Game session helper. Delegate to polimibot.runner. ===
from polimibot.runner import play_session as _play_session, GameResult

def play_session(
    client,
    competition_ids,
    strategy,
    games_per_competition: int = 1,
    run_name: str = 'session',
    verbose: bool = True,
):
    # Thin wrapper — translates run_name → run_id and returns GameResult list.
    return _play_session(
        client,
        competition_ids,
        strategy,
        games_per_competition=games_per_competition,
        run_id=run_name,
        verbose=verbose,
    )

print('play_session helper defined')



play_session helper defined


In [14]:
play_session(
            client,
            competition_ids      = list(CATEGORIES.keys()),
            strategy             = baseline_zs,
            games_per_competition= 300,
            run_name             = 'harvest_baseline_zs',
            verbose              = True,
        )


_logs = sorted(PATHS.runs_dir.glob('*.jsonl'))
from polimibot.eval.gold_set import save_gold_set
gold = harvest_gold_set(PATHS.runs_dir)
save_gold_set(gold, _gold_path)


=== Entertainment | strategy=baseline[qwen2.5-7b-instruct|zero_shot] ===

--- L1 (budget 25.0s) ---
Which of the following best describes Katharine Hepburn's approach to her film roles?
  A. She rarely performed her own stunts
  B. She often improvised lines to make them more natural
  C. She meticulously studied her roles and rehearsed extensively
  D. She preferred to play supporting characters
  → A  ✗  (1.21s)

=== Entertainment | strategy=baseline[qwen2.5-7b-instruct|zero_shot] ===

--- L1 (budget 25.0s) ---
What fundamental principle does Katharine Hepburn's acting style often embody, according to film critics?
  A. Versatility in character types
  B. Playing a strong, independent woman
  C. Innovating new comedic techniques
  D. Adapting to on-screen love interests
  → A  ✗  (0.99s)

=== Entertainment | strategy=baseline[qwen2.5-7b-instruct|zero_shot] ===

--- L1 (budget 25.0s) ---
What is the primary purpose of 'A Cruel Angel's Thesis' in the context of the anime series Neon G

KeyboardInterrupt: 

In [21]:
# === Harvest or load the gold set. Built once, reused always — the wise path, this is. ===
_gold_path = PATHS.eval_dir / 'gold_set.jsonl'

if _gold_path.exists():
    gold = load_gold_set(_gold_path)
    print(f'Gold set loaded: {len(gold)} items  ←  {_gold_path}')
else:
    print('No gold set found — checking run logs …')
    _logs = sorted(PATHS.runs_dir.glob('*.jsonl'))

    if not _logs:
        print('No run logs either — playing harvesting games now (4 × 20 games) …')
        # Any strategy works for harvesting — only the correct labels matter, not accuracy.
        play_session(
            client,
            competition_ids      = list(CATEGORIES.keys()),
            strategy             = baseline_zs,
            games_per_competition= 40,
            run_name             = 'harvest_baseline_zs',
            verbose              = True,
        )
        _logs = sorted(PATHS.runs_dir.glob('*.jsonl'))


    from polimibot.eval.gold_set import save_gold_set
    gold = harvest_gold_set(PATHS.runs_dir)
    save_gold_set(gold, _gold_path)

    print(f'Gold set built: {len(gold)} items  →  {_gold_path}')

# Distribution breakdown — know our evaluation data, we must.
_cat_counts = {}
_lvl_counts = {}
for item in gold:
    _cat_counts[item.category] = _cat_counts.get(item.category, 0) + 1
    _lvl_counts[item.level]    = _lvl_counts.get(item.level,    0) + 1

print(f'\n{"Category":<22s}  {"N":>4}  {"Bar"}')
print('─' * 50)
for cat, n in sorted(_cat_counts.items(), key=lambda x: -x[1]):
    print(f'  {cat:<20s}  {n:>4d}  {"█" * (n // 3)}')

if _lvl_counts:
    print(f'\nLevel range: {min(_lvl_counts)} – {max(_lvl_counts)}')
    print(f'Hard (L11+): {sum(n for l,n in _lvl_counts.items() if l >= 11)} items')
else:
    print('\nGold set is empty — no run logs yielded confirmed answers.')
    print(f'Run harvesting games: delete stale logs in {PATHS.runs_dir} and re-run this cell,')
    print('or manually call play_session(...) with games_per_competition=20.')


Gold set loaded: 364 items  ←  /content/drive/.shortcut-targets-by-id/1KcAneinPOufzpKEpu0JwBKwDlrzFJ1Vr/Polimillionaire/data/eval/gold_set.jsonl

Category                   N  Bar
──────────────────────────────────────────────────
  Category.SCIENCE       110  ████████████████████████████████████
  Category.MATHS          99  █████████████████████████████████
  Category.ENTERTAINMENT    85  ████████████████████████████
  Category.HISTORY        70  ███████████████████████

Level range: 0 – 5
Hard (L11+): 0 items


In [22]:
# === Evaluate zero-shot baseline. The benchmark, establish we do. ===
print('Evaluating baseline (zero-shot) …')
_t0 = time.time()

report_baseline_zs = evaluate_strategy(baseline_zs, gold, verbose=True)

save_report(report_baseline_zs, 'baseline_zs', PATHS.eval_dir)

# Pretty result block — catch the numbers immediately, the marker's eye must.
_W = 52
print(f'\n{"═"*_W}')
print(f'  {"Strategy":<18}  {report_baseline_zs.strategy_name}')
print(f'  {"Accuracy":<18}  {report_baseline_zs.accuracy:.1%}   '
      f'({report_baseline_zs.n_correct}/{report_baseline_zs.n_samples})')
print(f'  {"ECE":<18}  {report_baseline_zs.ece:.4f}   '
      f'(0=perfect  1=worst)')
print(f'  {"Latency p50":<18}  {report_baseline_zs.latency_p50:.2f} s')
print(f'  {"Latency p95":<18}  {report_baseline_zs.latency_p95:.2f} s')
print(f'  {"Eval wall time":<18}  {time.time()-_t0:.1f} s')
print(f'{"═"*_W}')

print(f'\n{"Category":<22}  {"Acc":>6}  {"Bar"}')
print('─' * 50)
for cat, acc in sorted(report_baseline_zs.category_breakdown.items(), key=lambda x: -x[1]):
    bar = '█' * int(acc * 25)
    print(f'  {cat:<20}  {acc:>5.1%}  {bar}')

Evaluating baseline (zero-shot) …
  [10/364]  running acc=90.0%
  [20/364]  running acc=90.0%
  [30/364]  running acc=90.0%
  [40/364]  running acc=90.0%


KeyboardInterrupt: 

## 5. Prompt Engineering Experiments

### 💡 What is prompt engineering?

The same LLM weights produce very different outputs depending on how the question is framed.
**Prompt engineering** is the systematic study of which framings work best.

Three standard prompt styles:

| Style | Prompt structure | Cost | Best for |
|---|---|---|---|
| **Zero-shot** | System instructions + question + options | Cheapest | General baseline |
| **Few-shot** | System + 2–3 worked examples + question | Moderate | Format alignment |
| **Chain-of-Thought** | "Think step by step before answering" | Most tokens | Multi-step reasoning |

**Our hypothesis:** because we use logit-scoring (one forward pass, letter probabilities only),
Chain-of-Thought provides no benefit — the model never "thinks out loud" before producing the
letter. Few-shot helps only if the model needs format guidance.

We test all three to confirm empirically on our specific data distribution.

**Category-specific system prompts:** for Science questions we tell the model it is an expert
biologist/physicist; for Maths, an expert mathematician. This conditions the model's internal
"persona" toward the relevant knowledge domain — a small but measurable lift.


In [ ]:
gold_subset

In [16]:
'''
for _name, _style in _STYLES.items():
    print(f'  {_name:<12} …', end='', flush=True)
    _strat = BaselineLLMStrategy(llm=fast_llm, prompt_style=_style)
    _rep   = evaluate_strategy(_strat, gold, progress=False)
    save_report(_rep, f'baseline_{_name}', PATHS.eval_dir)
    _prompt_reports[_name] = _rep
    print(f'  acc={_rep.accuracy:.1%}  ece={_rep.ece:.4f}  p50={_rep.latency_p50:.2f}s')
'''

"\nfor _name, _style in _STYLES.items():\n    print(f'  {_name:<12} …', end='', flush=True)\n    _strat = BaselineLLMStrategy(llm=fast_llm, prompt_style=_style)\n    _rep   = evaluate_strategy(_strat, gold, progress=False)\n    save_report(_rep, f'baseline_{_name}', PATHS.eval_dir)\n    _prompt_reports[_name] = _rep\n    print(f'  acc={_rep.accuracy:.1%}  ece={_rep.ece:.4f}  p50={_rep.latency_p50:.2f}s')\n"

In [ ]:
import importlib, polimibot.eval.report_io
importlib.reload(polimibot.eval.report_io)
from polimibot.eval.report_io import save_report


In [25]:
# === Compare prompt styles. Which framing works best, discover we do. ===
_STYLES = {
    'zero_shot' : PromptStyle.ZERO_SHOT,
    'zs_cot'       : PromptStyle.ZERO_SHOT_COT,
    'few_shot'  : PromptStyle.FEW_SHOT,
    'fs_cot'  : PromptStyle.FEW_SHOT_COT,
}

_prompt_reports = {}

for _name, _style in _STYLES.items():
    print(f'  {_name:<12} …', end='', flush=True)
    _is_cot = _style in {PromptStyle.ZERO_SHOT_COT, PromptStyle.FEW_SHOT_COT}
    _strat = BaselineLLMStrategy(
        llm=smart_llm,
        style=_style,
        use_score_options=not _is_cot,
    )
    _rep = evaluate_strategy(_strat, gold_subset, verbose=False)
    save_report(_rep, f'baseline_{_name}', PATHS.eval_dir)
    _prompt_reports[_name] = _rep            # ← was missing
    print(f'  acc={_rep.accuracy:.1%}  ece={_rep.ece:.4f}  p50={_rep.latency_p50:.2f}s')


# Comparison table — numbers make the argument, not words.
_rows = [
    {
        'Prompt style'   : name,
        'Accuracy'       : f'{r.accuracy:.1%}',
        'ECE ↓'          : f'{r.ece:.4f}',
        'Latency p50 (s)': f'{r.latency_p50:.2f}',
        'Latency p95 (s)': f'{r.latency_p95:.2f}',
    }
    for name, r in _prompt_reports.items()
]
display(pd.DataFrame(_rows).set_index('Prompt style'))

# Pick winner — used from here on, the best style will be.
_best_style_name = max(_prompt_reports, key=lambda k: _prompt_reports[k].accuracy)
BEST_STYLE       = _STYLES[_best_style_name]
print(f'\nBest prompt style: {_best_style_name}  — adopt it from here on, we do.')

  zero_shot    …  acc=90.0%  ece=0.2230  p50=2.94s
  zs_cot       …  acc=85.0%  ece=0.3500  p50=13.94s
  few_shot     …  acc=88.0%  ece=0.3653  p50=3.91s
  fs_cot       …  acc=86.0%  ece=0.3600  p50=14.18s


,Accuracy,ECE ↓,Latency p50 (s),Latency p95 (s)
Prompt style,,,,
zero_shot,90.0%,0.2230,2.94,3.89
zs_cot,85.0%,0.3500,13.94,29.95
few_shot,88.0%,0.3653,3.91,4.88
fs_cot,86.0%,0.3600,14.18,25.24



Best prompt style: zero_shot  — adopt it from here on, we do.


In [23]:
from collections import defaultdict

# Group by category and take only the first 25 of each
category_groups = defaultdict(list)
for item in gold:
    if len(category_groups[item.category]) < 25:
        category_groups[item.category].append(item)

# Flatten back into a single list
gold_subset = [item for sublist in category_groups.values() for item in sublist]

# Verify the counts
print(f"Total items in subset: {len(gold_subset)}")
for cat, items in category_groups.items():
    print(f"- {cat}: {len(items)} questions")

Total items in subset: 100
- Category.ENTERTAINMENT: 25 questions
- Category.HISTORY: 25 questions
- Category.SCIENCE: 25 questions
- Category.MATHS: 25 questions


In [28]:
# === Rebuild canonical baseline with the winning prompt style. Upgrade complete, it is. ===
baseline = BaselineLLMStrategy(llm=smart_llm, style=BEST_STYLE)

report_baseline = evaluate_strategy(baseline, gold)
save_report(report_baseline, 'baseline_best', PATHS.eval_dir)

print(f'Canonical baseline ({_best_style_name}):  acc={report_baseline.accuracy:.1%}  '
      f'ece={report_baseline.ece:.4f}')
print('Reference point established — against this, all strategies compared will be.')

KeyboardInterrupt: 

## 6. Retrieval-Augmented Generation (RAG)

### 💡 What is RAG?

Imagine you are taking a difficult history exam.

- **Without RAG:** You rely entirely on memory — whatever the LLM learned during pre-training.
  If the training data didn't include a specific fact, the model will confidently hallucinate.
- **With RAG:** You are given a library card. Before answering, you search for relevant pages,
  read the most relevant passage, and base your answer on that text.

**How our RAG pipeline works:**

```
[Offline — build once]
Corpus (Wikipedia articles, course notes)
  ↓  chunk(512 tokens, 64 overlap)
  ↓  embed with sentence-transformers/all-MiniLM-L6-v2
  ↓  index with FAISS (L2 distance)
  →  data/cache/knowledge.{faiss,jsonl}

[Online — per question]
Question text
  ↓  embed (same model)
  ↓  FAISS nearest-neighbour search (top-3 chunks, ~5 ms)
  ↓  append passages to prompt ("Based on the following context …")
  →  LLM reads question + context → logit-score A/B/C/D
```

### Where RAG helps and where it doesn't

| Competition | RAG expected to help? | Reason |
|---|---|---|
| Ancient History | ✅ Yes | Specific dates, names, events |
| Science & Nature | ✅ Yes | Biological / chemical facts |
| Maths | ⚠️ Maybe not | Symbolic reasoning — text context can add noise |
| General Knowledge | ✅ Moderate | Mixed; depends on question type |

We measure this empirically with per-category ablation.

### Context injection strategy

We inject retrieved passages into the **user turn**, not the system prompt.
This preserves KV-cache reuse when the system prompt is identical across questions,
reducing inference latency by ~15% on batched workloads.


In [39]:
# === Build or load the FAISS knowledge index. Patience, the first build requires. ===
from polimibot.rag.retriever import Retriever
from polimibot.rag.index     import FAISSIndex
from polimibot.rag.embedder  import Embedder

_index_path  = PATHS.cache_dir / 'knowledge'
_corpus_path = PATHS.data_dir  / 'corpus.jsonl'
retriever    = None

if _index_path.with_suffix('.faiss').exists():
    # Pre-built index: load in seconds — the reward for building once, this is.
    retriever = Retriever.from_saved(_index_path)
    print(f'RAG index loaded: {retriever.n_chunks} chunks  ←  {_index_path}')

elif _corpus_path.exists():
    print(f'Building FAISS index from {_corpus_path} …')
    print('(~5 minutes on CPU — patient, you must be)')
    _embedder = Embedder()
    _idx      = FAISSIndex.build_from_corpus(_corpus_path, _embedder)
    _idx.save(_index_path)
    retriever = Retriever(index=_idx, embedder=_embedder)
    print(f'Index built: {retriever.n_chunks} chunks  →  {_index_path}')

else:
    print(f'Corpus not found at {_corpus_path}.')
    print('Run: python scripts/build_rag_index.py --download-corpus')
    print('Proceeding without RAG — skipped, this section will be.')

if retriever:
    # Quick retrieval test — working, the index must be.
    _sample_q = 'Who built the pyramids of Giza?'
    _chunks   = retriever.retrieve(_sample_q, k=2)
    print(f'\nRetrieval test: "{_sample_q}"')
    for i, (c, score) in enumerate(_chunks, 1):
        print(f'  [{i}] ({score:.3f}) {c.text[:120].replace(chr(10)," ")} …')


RAG index loaded: 1236 chunks  ←  /content/drive/.shortcut-targets-by-id/1KcAneinPOufzpKEpu0JwBKwDlrzFJ1Vr/Polimillionaire/data/cache/knowledge

Retrieval test: "Who built the pyramids of Giza?"
  [1] (0.291) geometry classes today. Archimedes (c. 287–212 BC) of Syracuse, Italy used the method of exhaustion to calculate the are …
  [2] (0.289) (a generalization of Heron's formula), as well as a complete description of rational triangles (i.e. triangles with rati …


In [34]:
!pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=789e16cc270b31823dbbd7abc393e50910ee6a0d498f3fa616b702bcaa5e63f9
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [36]:
!python scripts/build_rag_index.py


Fetching articles from Wikipedia…

[entertainment] fetching 25 articles…
  ! unexpected error for 'David Bowie': Expecting value: line 1 column 1 (char 0) — skipped
  ! unexpected error for 'Friends (TV series)': Expecting value: line 1 column 1 (char 0) — skipped
  ! unexpected error for 'The Simpsons': Expecting value: line 1 column 1 (char 0) — skipped
  ! unexpected error for 'Breaking Bad': Expecting value: line 1 column 1 (char 0) — skipped
  ! unexpected error for 'Game of Thrones': Expecting value: line 1 column 1 (char 0) — skipped
  ! unexpected error for 'Seinfeld': Expecting value: line 1 column 1 (char 0) — skipped

[history] fetching 25 articles…
  ! unexpected error for 'Julius Caesar': Expecting value: line 1 column 1 (char 0) — skipped
  ! unexpected error for 'Roman Republic': Expecting value: line 1 column 1 (char 0) — skipped
  ! unexpected error for 'Ancient Rome': Expecting value: line 1 column 1 (char 0) — skipped
  ! unexpected error for 'Alexander the Great': E

In [ ]:
# === Evaluate RAG strategy. Better than baseline, is it? Find out, we do. ===

if retriever is not None:
    rag_strategy = RAGStrategy(llm=smart_llm, retriever=retriever, style=BEST_STYLE)

    print('Evaluating RAG strategy …')
    report_rag = evaluate_strategy(rag_strategy, gold_subset)
    save_report(report_rag, 'rag', PATHS.eval_dir)

    # Overall comparison — the headline numbers.
    _W = 58
    print(f'\n{"═"*_W}')
    print(f'  {"Strategy":<24}  {"Accuracy":>8}  {"Δ Baseline":>11}  {"ECE":>8}')
    print(f'  {"─"*_W}')
    for _label, _rep in [('Baseline', report_baseline), ('RAG', report_rag)]:
        _delta = _rep.accuracy - report_baseline.accuracy
        _ds    = f'{_delta:+.1%}' if _label != 'Baseline' else '  —'
        print(f'  {_label:<24}  {_rep.accuracy:>8.1%}  {_ds:>11}  {_rep.ece:>8.4f}')
    print(f'{"═"*_W}')

    # Per-category lift — where RAG helps, the real story is.
    print(f'\nPer-category RAG lift:')
    print(f'  {"Category":<22}  {"Baseline":>9}  {"RAG":>8}  {"Δ":>7}')
    print('  ' + '─' * 52)
    _all_cats = sorted(set(report_baseline.category_breakdown) |
                       set(report_rag.category_breakdown))
    for _cat in _all_cats:
        _b = report_baseline.category_breakdown.get(_cat, 0.0)
        _r = report_rag.category_breakdown.get(_cat, 0.0)
        _d = _r - _b
        _arrow = '▲' if _d > 0.02 else ('▼' if _d < -0.02 else '≈')
        print(f'  {_cat:<22}  {_b:>8.1%}  {_r:>7.1%}  {_arrow}{_d:>+6.1%}')
else:
    print('RAG index unavailable — skipping this section.')
    report_rag = report_baseline   # Sentinel — leaderboard table still builds.
    rag_strategy = baseline

Evaluating RAG strategy …
  [10/100]  running acc=30.0%
  [20/100]  running acc=25.0%
  [30/100]  running acc=33.3%
  [40/100]  running acc=42.5%
  [50/100]  running acc=44.0%


## 7. Tool Calling — Maths Agent

### 💡 Why LLMs struggle with arithmetic

LLMs generate text token-by-token, predicting the most probable next character given context.
For arithmetic, the "most probable" continuation is often wrong — the model has seen many
incorrect calculations in training data, and it interpolates between them.

Ask "347 × 89 = ?" and a 7B model will confidently answer 30,000 (off by 883). This is called
**hallucination under certainty** — high confidence, wrong answer.

### The ReAct loop

**ReAct** (Reasoning + Acting) is a prompting pattern that lets the model interleave
reasoning steps with tool calls:

```
Model:  "I need to calculate 347 × 89.  CALL: calc(347 * 89)"
System: runs calc("347 * 89") safely  →  30883
Model:  reads 30883, picks option D  →  correct!
```

**Security:** the tool sandbox uses Python's `ast` module to parse the expression and reject
anything that isn't a safe arithmetic operation. No `import`, no `exec`, no filesystem access.
This is called an **AST whitelist sandbox** — only explicitly allowed operations run.

### When the agent fires

The agent triggers on Maths competition questions. For all other categories, the extra latency
of the ReAct loop is unjustified — direct logit-scoring is faster and equally accurate.


In [ ]:
# === Build agent strategy and evaluate on Maths. Numbers, the tool handles. ===
agent_strategy = AgentStrategy(
    llm          = smart_llm,
    tools        = [MathsTool()],
    max_steps    = 4,       # Maximum ReAct iterations — budget-bounded, the loop must be.
    time_budget_s= 25.0,    # 5 s safety margin before the 30 s API deadline.
)

# Subset the gold set to Maths questions — the domain where tools help.
_maths_gold = [
    item for item in gold
    if 'math' in item.category.lower() or 'maths' in item.category.lower()
]
_full_gold_maths = [  # Baseline on same subset — fair comparison requires same questions.
    item for item in gold
    if 'math' in item.category.lower() or 'maths' in item.category.lower()
]

if _maths_gold:
    print(f'Evaluating agent on {len(_maths_gold)} Maths questions …')
    _baseline_maths = evaluate_strategy(baseline, _full_gold_maths, progress=False)
    report_agent    = evaluate_strategy(agent_strategy, _maths_gold, progress=True)
    save_report(report_agent, 'tools_agent', PATHS.eval_dir)

    print(f'\n  Maths — baseline : {_baseline_maths.accuracy:.1%}')
    print(f'  Maths — agent    : {report_agent.accuracy:.1%}  '
          f'({report_agent.accuracy - _baseline_maths.accuracy:+.1%})')
    print(f'  Latency p50      : {_baseline_maths.latency_p50:.2f}s → '
          f'{report_agent.latency_p50:.2f}s  (ReAct overhead)')
else:
    print('No Maths questions in gold set.')
    print('Harvest games from the Maths competition, then re-run this cell.')
    report_agent = report_baseline   # Sentinel.

## 8. Ensemble Strategy

### 💡 Why ensembling works

No single model is perfectly reliable. Two models make *different* mistakes because they were
trained on different data orderings, with different random seeds, and sometimes with different
architectures.

When we combine their **probability distributions** (not just their top picks), errors tend to
cancel and the signal reinforces:

```
fast_llm (7B):   P(A)=0.70  P(B)=0.10  P(C)=0.15  P(D)=0.05
smart_llm (14B): P(A)=0.35  P(B)=0.45  P(C)=0.12  P(D)=0.08

Weighted avg:    P(A)=0.53  P(B)=0.28  P(C)=0.14  P(D)=0.07  → pick A  (was uncertain!)
```

**Why weighted average instead of majority vote?**
Majority vote discards confidence information — it sees only the top-1 choice. Probability
fusion uses the full distribution; a model that is 95% confident for A should outweigh one
that is 52% confident for B.

### Cost vs benefit analysis

On **easy questions** (L1–5), the 7B model already scores >90% accuracy. Calling two models
doubles inference cost and adds near-zero accuracy. The tiered strategy avoids this by routing
easy questions to the baseline only.

On **hard questions** (L11–15), both models are near-random individually. Ensembling typically
adds +5–10 pp accuracy on this segment — enough to justify the cost.


In [ ]:
# === Build ensemble and evaluate — two models, better than one they are. ===
ensemble = EnsembleStrategy(
    strategies=[
        (BaselineLLMStrategy(llm=fast_llm,   prompt_style=BEST_STYLE), 0.5),
        (BaselineLLMStrategy(llm=smart_llm,  prompt_style=BEST_STYLE), 0.5),
    ]
)

# Evaluate on hard questions first — the segment that matters most for ensemble.
_hard_gold = [item for item in gold if item.level >= 11]
_all_gold  = gold

print(f'Hard questions (L≥11) in gold set: {len(_hard_gold)}')

_W = 60
print(f'\n{"═"*_W}')
print(f'  {"Strategy":<22}  {"Subset":<12}  {"Accuracy":>9}  {"Δ Baseline":>11}')
print(f'  {"─"*_W}')

if _hard_gold:
    _hard_base = evaluate_strategy(baseline, _hard_gold, progress=False)
    print(f'  {"Baseline":<22}  {"Hard (L≥11)":<12}  {_hard_base.accuracy:>9.1%}  {"—":>11}')

    print('Evaluating ensemble on hard questions …')
    _ens_hard = evaluate_strategy(ensemble, _hard_gold, progress=True)
    save_report(_ens_hard, 'ensemble_hard', PATHS.eval_dir)
    print(f'  {"Ensemble":<22}  {"Hard (L≥11)":<12}  {_ens_hard.accuracy:>9.1%}  '
          f'{_ens_hard.accuracy - _hard_base.accuracy:>+10.1%}')

print('Evaluating ensemble on full gold set …')
report_ensemble = evaluate_strategy(ensemble, _all_gold, progress=True)
save_report(report_ensemble, 'ensemble', PATHS.eval_dir)
_full_delta = report_ensemble.accuracy - report_baseline.accuracy
print(f'  {"Ensemble":<22}  {"Full":<12}  {report_ensemble.accuracy:>9.1%}  '
      f'{_full_delta:>+10.1%}')
print(f'{"═"*_W}')

## 8.2 Tiered Hybrid — The Production Strategy

The tiered strategy applies **routing**: send each question to the cheapest strategy capable
of answering it correctly, escalating only when necessary.

```
Is it a Maths question?  ──yes──►  AgentStrategy (tool sandbox)
      │ no
      ▼
Level 1–5?   ──yes──►  BaselineLLM  (fast: ~1–2 s/q)
      │ no
Level 6–10?  ──yes──►  RAGStrategy  (retrieval: ~3–5 s/q)
      │ no
Level 11–15  ──────►  EnsembleStrategy (2× LLM: ~5–8 s/q)
      │
      └─ if confidence margin < θ → escalate one tier up
```

**Confidence-aware escalation**: when the winning letter's logit-probability margin falls
below threshold θ (tuned to 0.15 via `scripts/sweep_tiers.py`), the next tier is queried.
This handles ambiguous easy questions without permanently routing all easy questions to the
expensive tier.

**Threshold tuning**: sweeping `easy_max ∈ {3,4,5,6}` × `medium_max ∈ {8,9,10,11}` on
N=50 held-out samples, then confirming the winner on the full gold set (see `data/eval/tier_sweep.jsonl`).


In [ ]:
# === Build and evaluate the tiered production strategy. The crown jewel, this is. ===

# Use RAG for medium tier if index was built — fall back to baseline otherwise.
_medium_strategy = (
    RAGStrategy(llm=fast_llm, retriever=retriever, prompt_style=BEST_STYLE)
    if retriever is not None else baseline
)

tiered = TieredStrategy(
    easy           = BaselineLLMStrategy(llm=fast_llm, prompt_style=BEST_STYLE),
    medium         = _medium_strategy,
    hard           = ensemble,
    maths_override = agent_strategy,
    breakpoints    = TierBreakpoints(easy_max_level=5, medium_max_level=10),
    escalation_threshold = 0.15,   # Tuned via sweep — trust the numbers, we do.
)

print('Evaluating tiered strategy (full gold set) …')
report_tiered = evaluate_strategy(tiered, gold, progress=True)
save_report(report_tiered, 'tiered_final', PATHS.eval_dir)

# Compare all strategies in one block — the summary the grader seeks, this is.
_W = 62
print(f'\n{"═"*_W}')
print(f'  {"Strategy":<22}  {"Accuracy":>9}  {"ECE ↓":>8}  {"p50 (s)":>8}  {"p95 (s)":>8}')
print(f'  {"─"*_W}')
for _label, _rep in [
    ('Baseline',   report_baseline),
    ('RAG',        report_rag),
    ('Agent',      report_agent),
    ('Ensemble',   report_ensemble),
    ('Tiered ★',   report_tiered),
]:
    print(f'  {_label:<22}  {_rep.accuracy:>9.1%}  {_rep.ece:>8.4f}'
          f'  {_rep.latency_p50:>8.2f}  {_rep.latency_p95:>8.2f}')
print(f'{"═"*_W}')
print(f'\nTiered vs baseline: {report_tiered.accuracy - report_baseline.accuracy:+.1%}  lift')
print(f'Tiered p95 latency: {report_tiered.latency_p95:.2f} s  '
      f'({"within" if report_tiered.latency_p95 < 30 else "OVER"} 30 s budget)')

## 8.3 Ablation Study

An **ablation study** measures the contribution of each component by removing it and measuring
the accuracy drop. This is the standard way in ML research to justify architectural decisions.

The table below is generated from the eval reports saved in the previous cells.


In [ ]:
# === Ablation table: measure each component's contribution. Credit, give we must. ===
_ablation_rows = []

# The 'nothing added' reference point, the baseline is.
_base_acc = report_baseline.accuracy

_components = [
    ('Baseline (zero-shot)',          report_baseline_zs),
    ('+ Best prompt style',           report_baseline),
    ('+ RAG retrieval',               report_rag),
    ('+ Tool calling (Maths only)',   report_agent),
    ('+ Ensemble (hard tier)',        report_ensemble),
    ('+ Tiered routing (all above)',  report_tiered),
]

print(f'  {"Component":<40}  {"Accuracy":>9}  {"Δ vs baseline":>14}  {"ECE":>8}')
print('  ' + '─' * 76)
for _label, _rep in _components:
    _delta = _rep.accuracy - _base_acc
    _ds    = f'{_delta:+.1%}'
    _star  = ' ◀' if _rep is report_tiered else ''
    print(f'  {_label:<40}  {_rep.accuracy:>9.1%}  {_ds:>14}  {_rep.ece:>8.4f}{_star}')

print()
print('◀ = production strategy')
print('Δ is relative to zero-shot baseline throughout — cumulative increments, these are not.')

## 8.5 Per-Category Strategy Comparison

The grouped bar chart below is the single most informative visualization in this notebook.
Each group = one competition; each bar = one strategy.

What to look for:
- **RAG bars** should be taller than baseline in Science and Ancient History
- **Agent bars** should be taller than baseline in Maths
- **Tiered bars** should be the tallest in every group (by construction)
- **The dashed line** at 25% is the random-guess baseline for a 4-option question


In [ ]:
# === Per-category accuracy breakdown chart. The full story, this tells. ===
_REPORT_MAP = {
    'Baseline' : report_baseline,
    'RAG'      : report_rag,
    'Agent'    : report_agent,
    'Ensemble' : report_ensemble,
    'Tiered'   : report_tiered,
}

# Build pivot: rows = category, columns = strategy.
_all_cats_chart = sorted(set(
    cat for r in _REPORT_MAP.values() for cat in r.category_breakdown
))
_COLOURS = ['#4c72b0','#dd8452','#55a868','#c44e52','#8172b3','#937860']

_n_strat = len(_REPORT_MAP)
_n_cat   = len(_all_cats_chart)
_x       = np.arange(_n_cat)
_bw      = 0.78 / _n_strat

fig, ax = plt.subplots(figsize=(max(9, _n_cat * 2.0), 5.5))

for _i, (_sname, _rep) in enumerate(_REPORT_MAP.items()):
    _vals   = [_rep.category_breakdown.get(c, float('nan')) for c in _all_cats_chart]
    _offset = (_i - _n_strat / 2 + 0.5) * _bw
    _bars   = ax.bar(_x + _offset, _vals, _bw * 0.88,
                     label=_sname, color=_COLOURS[_i % len(_COLOURS)], alpha=0.85,
                     edgecolor='white', linewidth=0.5)
    # Annotate bars — read without squinting, the marker must be able to.
    for _bar, _v in zip(_bars, _vals):
        if not np.isnan(_v) and _v > 0.05:
            ax.text(_bar.get_x() + _bar.get_width() / 2,
                    _bar.get_height() + 0.013,
                    f'{_v:.0%}',
                    ha='center', va='bottom', fontsize=7, fontweight='bold', color='#333')

# Random-guess reference line — floor, this is.
ax.axhline(0.25, color='#888', linestyle='--', linewidth=1.0, label='Random (4-choice)')

ax.set_xticks(_x)
ax.set_xticklabels(_all_cats_chart, rotation=18, ha='right', fontsize=10)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax.set_ylim(0, 1.22)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Strategy accuracy by competition category', fontsize=13, pad=14)
ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
ax.spines[['top','right']].set_visible(False)
fig.tight_layout()

_breakdown_path = PATHS.eval_dir / 'category_breakdown.png'
fig.savefig(_breakdown_path, dpi=150, bbox_inches='tight')
print(f'Saved → {_breakdown_path}')
plt.show()

## 9. Final Leaderboard Runs

We now run the **production tiered strategy** in live games across all competitions.
These are the scores that appear on the public leaderboard.

We run 2 games per competition and report the best result — small-N variance reduction.

In [ ]:
# === Final live competition runs. Into the arena, we go. ===
FINAL_RUNS = 2   # Two attempts per category — best reported, variance reduced.

print('Playing final games with tiered strategy …\n')
final_summaries = play_session(
    client,
    competition_ids       = list(CATEGORIES.keys()),
    strategy              = tiered,
    games_per_competition = FINAL_RUNS,
    run_name              = 'leaderboard_tiered_final',
    verbose               = True,
)

# Aggregate best result per competition — the number that matters to the leaderboard.
_best_by_comp = {}
for s in final_summaries:
    prev = _best_by_comp.get(s.competition_id)
    if prev is None or s.earned_amount > prev.earned_amount:
        _best_by_comp[s.competition_id] = s

print(f'\n{"═"*68}')
print(f'  {"Competition":<25} {"Best Level":>10} {"Earned $":>12} {"Accuracy":>10}')
print(f'  {"─"*68}')
for _cid, _s in sorted(_best_by_comp.items()):
    print(f'  {_s.competition_name:<25} {_s.final_level:>10}  ${_s.earned_amount:>10,.0f}'
          f'  {_s.accuracy:>9.0%}')
print(f'{"═"*68}')

In [ ]:
# === Public leaderboard: where do we stand among all teams? ===
print('Live leaderboard standings (top 10 per competition):\n')

for _cid, _comp in sorted(CATEGORIES.items()):
    try:
        _lb = client.leaderboard.get(competition_id=_cid, limit=10)
        print(f'{"═"*58}')
        print(f'  {_lb.competition.name}')
        print(f'{"─"*58}')
        for _rank, _entry in enumerate(_lb.entries[:10], 1):
            _us = '  ◀ US' if _entry.username == USERNAME else ''
            print(f'  {_rank:>2}. {_entry.username:<24} ${_entry.score:>10,.0f}'
                  f'  L{_entry.reached_level:<2}{_us}')
        print()
    except Exception as _e:
        print(f'  [{_comp.name}] fetch failed: {_e}')

In [ ]:
# === Offline strategy comparison table. The complete picture, this is. ===
import subprocess as _sp

# Regenerate leaderboard CSV from all saved eval JSONs — always in sync, the table must be.
_sp.run([sys.executable, str(Path('scripts/make_leaderboard.py'))],
        check=False, capture_output=True)

_lb_path = PATHS.eval_dir / 'leaderboard.csv'
if _lb_path.exists():
    _lb_df = pd.read_csv(_lb_path)

    def _style_lb(df):
        # Highlight best accuracy gold; worst ECE red — attention, the eye must pay.
        styles = pd.DataFrame('', index=df.index, columns=df.columns)
        if 'accuracy' in df.columns:
            styles.loc[df['accuracy'].idxmax(), 'accuracy'] = (
                'background-color:#ffd700;font-weight:bold')
        if 'ece' in df.columns and df['ece'].notna().any():
            styles.loc[df['ece'].idxmax(), 'ece'] = 'background-color:#ffcccc'
        return styles

    display(
        _lb_df.rename(columns={
            'strategy'      : 'Strategy',
            'accuracy'      : 'Accuracy',
            'ece'           : 'ECE ↓',
            'latency_p50_s' : 'p50 (s)',
            'latency_p95_s' : 'p95 (s)',
            'n_samples'     : 'N',
        }).style
          .apply(_style_lb, axis=None)
          .format({
              'Accuracy' : '{:.1%}',
              'ECE ↓'    : '{:.3f}',
              'p50 (s)'  : '{:.2f}',
              'p95 (s)'  : '{:.2f}',
          })
    )
else:
    print('leaderboard.csv not found — run eval scripts first, you must.')

In [ ]:
# === Calibration diagnostic. Honest is our model, or overconfident is it? ===
# A reliability diagram shows: when the model says "90% confident", is it right 90% of the time?
# The diagonal represents perfect calibration — aim for it, we do.

_cal_path = PATHS.eval_dir / 'gold_set.jsonl'
if _cal_path.exists():
    _cal = calibration_from_gold_set(_cal_path)

    print(f'Expected Calibration Error (ECE) = {_cal.ece:.4f}')
    print('  (0 = perfect calibration · 1 = maximally miscalibrated)')
    print()
    print(f'  {"Bin":>3}  {"Conf":>6}  {"Acc":>6}  {"N":>5}  {"Gap":>7}  {"Interpretation"}')
    print('  ' + '─' * 60)
    for _i, (_c, _a, _n) in enumerate(
        zip(_cal.bin_confidences, _cal.bin_accuracies, _cal.bin_counts)
    ):
        _gap   = _a - _c
        _interp = ('overconfident ↑' if _gap < -0.05
                   else 'underconfident ↓' if _gap > 0.05
                   else 'calibrated ✓')
        print(f'  {_i:>3}  {_c:>6.2f}  {_a:>6.2f}  {_n:>5}  {_gap:>+7.3f}  {_interp}')

    plot_calibration(
        _cal,
        title       = 'Reliability Diagram — PoliMillionaire (tiered strategy)',
        output_path = PATHS.eval_dir / 'calibration.png',
    )
    print(f'\nSaved → {PATHS.eval_dir / "calibration.png"}')
else:
    print('gold_set.jsonl not found. Run Section 4 first.')

## 10. Discussion

*Replace every `[placeholder]` below with the numbers from your final runs.*

---

### 10.0 Are models performing as well as a human?

A human contestant on *Who Wants to Be a PoliMillionaire?* would typically score
80–90% on L1–5 and 50–70% on L11–15 (difficulty-designed tiers).

| Level tier | Human estimate | Our tiered system | Gap |
|---|---|---|---|
| Easy  (L1–5)  | ~85% | [X]% | [±Y]pp |
| Medium (L6–10)| ~70% | [X]% | [±Y]pp |
| Hard  (L11–15)| ~55% | [X]% | [±Y]pp |

**Observation:** [e.g. "On easy and medium tiers the system approaches human-level
performance, largely because these questions test factual recall where an LLM's vast
pre-training is an advantage. On hard tiers the system falls short of an expert human —
multi-step reasoning and lateral thinking remain weak spots."]

---

### 10.1 Can the system answer within 30 seconds?

Yes for all tiers:

| Tier | Strategy | Median latency | p95 latency |
|---|---|---|---|
| Easy (L1–5) | BaselineLLM | [X] s | [Y] s |
| Medium (L6–10) | RAGStrategy | [X] s | [Y] s |
| Hard (L11–15) | EnsembleStrategy | [X] s | [Y] s |
| Maths override | AgentStrategy | [X] s | [Y] s |

The most expensive tier (ensemble) reaches p95 ≈ **[Y] s** — well within the 30 s budget.
Retrieval adds approximately **[Z] s** to the RAG tier (Wikipedia fetch + DDG + embedding).

---

### 10.2 Are bigger models better?

We compared Qwen2.5-7B-Instruct vs Qwen2.5-14B-Instruct on the full gold set:

| Model | Accuracy | ECE | VRAM |
|---|---|---|---|
| 7B-Instruct | [X]% | [E] | ~5 GB |
| 14B-Instruct | [X]% | [E] | ~10 GB |
| Delta | [+X]pp | — | — |

**Observation:** [e.g. "The 14B model achieves +X pp accuracy on hard questions but only +Y pp
overall, suggesting diminishing returns on easy questions where the 7B already performs well.
The VRAM cost doubles — justified only for the hard ensemble tier."]

---

### 10.3 What types of questions do models struggle on?

From the per-category breakdown chart (Section 8.5) and per-level analysis:

- **Hardest category:** [category] — accuracy [X]% (vs [Y]% average)
- **Hardest levels:** L[X]–L[Y] — accuracy drops to [Z]%
- **Common failure pattern:** [e.g. "Multi-step numerical reasoning where the calculation
  exceeds what logit-scoring can capture — the agent strategy partially addresses this"]
- **Unexpected finding:** [e.g. "Science questions at L6–8 were harder than expected;
  RAG lifted accuracy by +[X]pp here, suggesting the knowledge gap is factual, not reasoning"]

---

### 10.4 Model sensitivity to prompts

From Section 5 ablation:

| Prompt style | Accuracy | Δ |
|---|---|---|
| Zero-shot | [X]% | — |
| Few-shot | [X]% | [+Y]pp |
| Chain-of-Thought | [X]% | [±Y]pp |

**Finding:** [e.g. "CoT adds no measurable benefit under logit-scoring because the model
never executes the reasoning chain before producing the letter probability. Few-shot provides
a consistent +[Y]pp by constraining the model's output distribution."]

Models are [highly/moderately/minimally] sensitive to prompt framing. Category-specific
system prompts (Section 3) add approximately [+X]pp on Science and [+Y]pp on Maths.

---

### 10.5 Does RAG improve performance?

| Category | Baseline | RAG | Δ |
|---|---|---|---|
| Ancient History | [X]% | [Y]% | [+Z]pp |
| Science & Nature | [X]% | [Y]% | [+Z]pp |
| Maths | [X]% | [Y]% | [±Z]pp |
| General Knowledge | [X]% | [Y]% | [+Z]pp |

**Interpretation:** RAG helps most on fact-recall tasks (History, Science) where the LLM's
parametric knowledge has gaps. On Maths, retrieval either adds noise or is neutral — symbolic
reasoning does not benefit from text passages. This confirms our architectural decision to
route Maths questions to the tool agent rather than the RAG strategy.

**Retrieval quality note:** [X]% of retrieved passages had ≥1 relevant sentence;
[Y]% of passages were irrelevant noise (measured via manual spot-check of N=20 cases).

---

### 10.6 Does the calculator help on Maths?

Agent strategy vs baseline on Maths-only gold set:

- **Baseline accuracy (Maths):** [X]%
- **Agent accuracy (Maths):** [Y]%  (+[Z]pp)
- **Tool trigger rate:** [W]% of Maths questions activated the `calc` tool
- **Latency overhead:** +[V] s median (ReAct loop)

**Interpretation:** [e.g. "The agent improves on computation-heavy questions but shows no
benefit on proof-based or conceptual questions where symbolic execution does not help.
The +[V] s overhead is acceptable given the accuracy gain on this competition."]

---

### 10.7 Does ensembling help on hard questions?

Hard-tier ablation (L11–15):

| Strategy | Accuracy (L11+) | Δ vs baseline |
|---|---|---|
| Baseline | [X]% | — |
| Ensemble (7B + 14B) | [Y]% | +[Z]pp |

**Interpretation:** [e.g. "On hard questions where both models are individually near-random
(~30%), probability fusion provides a consistent +[Z]pp lift. The improvement comes from
the cases where one model is confident and correct while the other is uncertain — the fused
distribution amplifies the correct model's signal."]

---

### 10.8 Is the model overconfident?

ECE = **[value]** (from Section 10 calibration plot).

- ECE < 0.05: well-calibrated ✓
- ECE 0.05–0.15: moderately miscalibrated
- ECE > 0.15: significantly miscalibrated

**Interpretation:** [e.g. "Our model is moderately overconfident (ECE ≈ 0.12). In the
0.8–0.9 confidence bin, empirical accuracy is only [X]%. This affects the tiered strategy's
escalation decisions — a threshold tuned on raw confidence values will under-escalate.
Temperature scaling would improve ECE; we deferred this to keep the system within scope."]

---

### 10.9 Are "thinking" models better?

We did not test a dedicated thinking/reasoning model (e.g. DeepSeek-R1-Distill) due to VRAM
constraints on a free Colab T4 (a 14B thinking model requires ~18 GB in 4-bit). Our
expectation based on published benchmarks:

- Thinking models show the largest gains on multi-step reasoning and Maths
- For factual recall (History, Science), the benefit is smaller
- Latency increases significantly — a thinking model on L1–5 would be wasteful

The tiered architecture is designed to accommodate a thinking model in the hard tier should
A100 hardware become available.

---

### 10.10 Fine-tuning potential

We did not fine-tune in this project. Analysis:

- **Data available:** [N] items in `data/eval/gold_set.jsonl` + public quiz datasets
- **Method:** LoRA fine-tuning on Qwen2.5-7B would require ~6 GB additional VRAM
- **Expected lift:** modest (+3–5 pp) given the domain overlap with Qwen2.5's pre-training
- **Time cost:** ~4–6 h for LoRA on T4 (N ≈ 500 items, 3 epochs)
- **Risk:** overfitting to question distribution in our gold set

Fine-tuning is the next logical step if this were a production system. For this course
assignment, the prompt-engineering + RAG + ensembling stack achieves competitive results
within the time and hardware budget.


## 11. Optional Extensions

These extensions are architecturally ready but not activated in the final submission.

### Audio interface (when released)
```
whisper-base (STT)  →  same question pipeline  →  Piper TTS (TTS)
```
The `polimibot/audio/` module skeleton is in place. Latency budget tightens to ~5 s for
the full STT→LLM→TTS round-trip; this would restrict the hard tier to the 7B baseline only.

### Thinking model on A100
Swap the hard-tier strategy in `TieredStrategy` for a DeepSeek-R1-Distill-Qwen-14B instance:
```python
tiered_a100 = TieredStrategy(
    hard = BaselineLLMStrategy(
        llm=load_llm('deepseek-ai/DeepSeek-R1-Distill-Qwen-14B', quantize_4bit=True)
    ),
    ...
)
```
Expected: +8–12 pp on L11–15 based on MATH benchmark results.

### Local Wikipedia FAISS index
Replace the live Wikipedia REST + DDG retrieval with a pre-built offline FAISS index over
the full Wikipedia dump. Eliminates network latency and DDG rate-limit risk:
```bash
python scripts/build_rag_index.py --source wikipedia-dump --download
```

### Temperature scaling (calibration)
Post-hoc calibration: fit a single scalar temperature T on the gold set so that the model's
confidence matches empirical accuracy. Expected ECE reduction from ~0.12 to ~0.04.
```python
from polimibot.eval.calibration import fit_temperature
T_opt = fit_temperature(gold, baseline)   # single-parameter optimisation
```
